# Pipeline Comparison: Baseline vs Extended

**Question:** For each classifier, does the *extended* pipeline (with hair removal) outperform the *baseline* pipeline (no hair removal)?

**Method:**
- **Cross-validation AUC**: 5 folds per (classifier, pipeline)
- **Held-out test-set ROC**: single AUC per (classifier, pipeline)
- **Paired tests** (Wilcoxon signed-rank + paired t-test) on fold-level AUCs

Both pipelines use the **same patient-level fold split** (`random_state=42`), so fold *i* contains the same patients in both pipelines - the comparison is properly paired.

## 0. Imports & setup

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import roc_curve, roc_auc_score
from scipy.stats import wilcoxon, ttest_rel

FIG_DIR = './results/figures'
os.makedirs(FIG_DIR, exist_ok=True)

## 1. Load predictions and CV-fold AUCs for both pipelines

In [ ]:
MODELS    = ['DT', 'kNN', 'LR']
PIPELINES = ['baseline', 'extended']

preds, cv = {}, {}
for m in MODELS:
    for p in PIPELINES:
        preds[(m, p)] = pd.read_csv(f'./results/predictions/predictions_{m}_{p}.csv')
        cv[(m, p)]    = pd.read_csv(f'./results/predictions/cv_folds_{m}_{p}.csv')

print('Loaded:')
for k in preds: print(f'  predictions{k}: {len(preds[k])} rows')
for k in cv:    print(f'  cv_folds   {k}: {len(cv[k])} folds')

## 2. Comparison table

Per classifier: mean CV AUC for both pipelines, mean paired difference, and significance tests.

In [ ]:
rows = []
for m in MODELS:
    base = cv[(m, 'baseline')]['auc'].values
    ext  = cv[(m, 'extended')]['auc'].values
    diff = ext - base
    try:
        _, w_p = wilcoxon(diff)
    except ValueError:
        w_p = np.nan
    _, t_p = ttest_rel(ext, base)
    rows.append({
        'Model':           m,
        'AUC baseline':    base.mean(),
        'AUC extended':    ext.mean(),
        'Mean diff':       diff.mean(),
        'Std baseline':    base.std(ddof=1),
        'Std extended':    ext.std(ddof=1),
        'Wilcoxon p':      w_p,
        'Paired t-test p': t_p,
    })
comparison_df = pd.DataFrame(rows)
comparison_df.round(4)

## 3. Bar chart - mean CV AUC with std error bars

In [ ]:
x = np.arange(len(MODELS))
width = 0.35

base_means = [cv[(m, 'baseline')]['auc'].mean() for m in MODELS]
base_stds  = [cv[(m, 'baseline')]['auc'].std(ddof=1) for m in MODELS]
ext_means  = [cv[(m, 'extended')]['auc'].mean() for m in MODELS]
ext_stds   = [cv[(m, 'extended')]['auc'].std(ddof=1) for m in MODELS]

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width/2, base_means, width, yerr=base_stds, label='baseline', capsize=5, color='tab:blue')
ax.bar(x + width/2, ext_means,  width, yerr=ext_stds,  label='extended', capsize=5, color='tab:orange')
ax.set_xticks(x)
ax.set_xticklabels(MODELS)
ax.set_ylabel('Mean CV AUC (error bars: 1 std)')
ax.set_title('Cross-validation AUC: baseline vs extended')
ax.set_ylim(min(min(base_means), min(ext_means)) - 0.05,
            max(max(base_means), max(ext_means)) + 0.05)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/pipeline_comparison_bars.png', dpi=150)
plt.show()

## 4. Paired per-fold dot plot

Each fold is one paired observation. Lines connect the same fold across pipelines:
- **Red** line: extended did *worse* than baseline on this fold
- **Green** line: extended did *better*

In [ ]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(15, 4), sharey=True)
for ax, m in zip(axes, MODELS):
    base = cv[(m, 'baseline')]['auc'].values
    ext  = cv[(m, 'extended')]['auc'].values
    ax.scatter([0]*len(base), base, color='tab:blue',   s=80, zorder=3, label='baseline')
    ax.scatter([1]*len(ext),  ext,  color='tab:orange', s=80, zorder=3, label='extended')
    for i in range(len(base)):
        col = 'tab:red' if ext[i] < base[i] else 'tab:green'
        ax.plot([0, 1], [base[i], ext[i]], color=col, alpha=0.5, lw=1.5)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['baseline', 'extended'])
    ax.set_title(f'{m}  (n={len(base)} folds)')
    if m == MODELS[0]:
        ax.set_ylabel('AUC')
    ax.grid(axis='y', alpha=0.3)
axes[0].legend(loc='lower left', fontsize=8)
plt.suptitle('Per-fold AUC: baseline vs extended (red line = extended worse)', y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/pipeline_comparison_paired.png', dpi=150)
plt.show()

## 5. ROC overlay (held-out test set)

One subplot per classifier, with baseline and extended ROC curves overlaid.

In [ ]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(15, 5))
for ax, m in zip(axes, MODELS):
    for p, color in [('baseline', 'tab:blue'), ('extended', 'tab:orange')]:
        df = preds[(m, p)]
        fpr, tpr, _ = roc_curve(df['label'], df['probability'])
        auc = roc_auc_score(df['label'], df['probability'])
        ax.plot(fpr, tpr, color=color, lw=2, label=f'{p} (AUC={auc:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
    ax.set_xlabel('False Positive Rate')
    if m == MODELS[0]:
        ax.set_ylabel('True Positive Rate')
    ax.set_title(m)
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(alpha=0.3)
plt.suptitle('ROC: baseline vs extended (held-out test set)', y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/pipeline_comparison_roc.png', dpi=150)
plt.show()

## Summary and decision

**Cross-validation**: For every classifier the extended pipeline produced a slightly lower mean CV AUC than baseline, and the direction (`extended - baseline < 0`) was consistent across all three models. With n = 5 folds, the paired Wilcoxon test cannot reach p < 0.05 even in principle (minimum two-sided p = 0.0625); the paired t-test also did not reject the null. The absence of significance therefore reflects **low statistical power**, not evidence that the pipelines are equivalent.

**Held-out test set**: ROC curves for baseline and extended overlap closely for every classifier, with baseline marginally above or equal. No classifier benefited from hair removal.

**Decision - use the baseline pipeline.** When two configurations are statistically indistinguishable, Occam's razor favours the simpler one. Baseline performs at least as well as extended for every classifier, runs faster (no hair-removal step), and removes the dependency on hair annotations. Together with the upstream feature-level analysis (`feature_comparison.ipynb`), which showed that hair removal alters HSV colour features without improving downstream performance, the choice of baseline is well supported.